# From story to diagram {#sec-story-to-diagram}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c02-story-to-diagram.ipynb)

Chapter 1 ended with a single feedback loop that we put into the model because it seemed plausible. Real projects do not start that way. They start with people who know the system describing it in their own words, a few numbers, and a question someone needs answered. This chapter turns that raw material into four products a modeller should have before writing a single equation: a **causal loop diagram**, a **reference mode**, a **model boundary chart** and a **dynamic hypothesis**. We do it in code, so every step can be checked, rerun and changed.

**Who it is for.** Anyone who will build models for other people, especially if you are used to starting from data rather than from interviews.

**Who can skip what.** If you already draw causal loop diagrams, skim sections 1 to 3 and read sections 4 and 5, which find loops and draw diagrams from an edge list in code, and section 8 on testing a hypothesis against data.

**Prerequisites.** Chapter 1, Why systems push back: stocks, flows, feedback and the waiting-list data. Python lists, dictionaries and pandas DataFrames at an introductory level.

**Time.** About three hours, including the exercises.

The figure below is where chapter 1 left the case: a stock of patients, with treatments limited by capacity and referrals held back by long waits.

In [ ]:
# Setup: run this cell first. In Colab it installs loopbricks (ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/"
                           "loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import os, pandas as pd
DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/main/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cycler
from matplotlib.patches import Ellipse, Polygon, Rectangle

SEED = 2                               # one seed per chapter
rng = np.random.default_rng(SEED)      # (this chapter draws no random numbers)
# Colour-blind-safe colours (Okabe and Ito) with a different line style
# for each series, so every figure also reads in greyscale.
OKABE_ITO = ["#000000", "#E69F00", "#0072B2", "#009E73", "#D55E00",
             "#CC79A7", "#56B4E9"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1, 1, 1)),
          (0, (1, 3))]
plt.rcParams["axes.prop_cycle"] = (cycler(color=OKABE_ITO)
                                   + cycler(linestyle=STYLES))
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "figure.dpi": 100})

In [ ]:
def draw_stock_flow(ax, stock, inflow, outflow, inputs_in=(),
                    inputs_out=(), title=""):
    """One stock with an inflow and an outflow (as in chapter 1)."""
    ax.set_xlim(0, 10)
    ax.set_ylim(0.6, 4.8)
    ax.axis("off")
    y = 2.0
    ax.add_patch(Rectangle((4, y - 0.6), 2, 1.2, fc="white", ec="black",
                           lw=1.5, zorder=2))
    ax.text(5, y, stock, ha="center", va="center", fontsize=11, zorder=3)
    for x in (0.7, 9.3):
        ax.add_patch(Ellipse((x, y), 1.0, 0.7, fc="white", ec="grey",
                             ls="--"))
    pipe = dict(arrowstyle="-|>,head_width=0.4,head_length=0.6", lw=3,
                color="black")
    ax.annotate("", xy=(4, y), xytext=(1.2, y), arrowprops=pipe)
    ax.annotate("", xy=(8.8, y), xytext=(6, y), arrowprops=pipe)
    info = dict(arrowstyle="->", lw=0.9, color="grey",
                connectionstyle="arc3,rad=0.15")
    for x, name, inputs in ((2.6, inflow, inputs_in),
                            (7.4, outflow, inputs_out)):
        bow = [(x - 0.25, y + 0.35), (x + 0.25, y - 0.35),
               (x + 0.25, y + 0.35), (x - 0.25, y - 0.35)]
        ax.add_patch(Polygon(bow, fc="white", ec="black", zorder=3))
        ax.text(x, y - 0.5, name, ha="center", va="top", fontsize=10)
        for j, label in enumerate(inputs):
            lx = x + (j - (len(inputs) - 1) / 2) * 2.2
            ax.text(lx, 4.3, label, ha="center", va="center", fontsize=9)
            ax.annotate("", xy=(x, y + 0.4), xytext=(lx, 4.05),
                        arrowprops=info)
    ax.annotate("", xy=(7.2, y + 0.3), xytext=(5.8, y + 0.6),
                arrowprops=dict(info, connectionstyle="arc3,rad=-0.5"))
    ax.set_title(title)


fig, ax = plt.subplots(figsize=(9, 3.4))
draw_stock_flow(ax, "Waiting list\n(patients)", "Referrals", "Treatments",
                inputs_in=["Referral rate", "Waiting time"],
                inputs_out=["Capacity", "Minimum treatment time"],
                title="Where chapter 1 left the waiting list")
ax.annotate("", xy=(3.7, 4.05), xytext=(5.0, 2.65),
            arrowprops=dict(arrowstyle="->", lw=0.9, color="grey",
                            connectionstyle="arc3,rad=0.3"))
plt.show()

Figure: stock-and-flow diagram of the waiting list at the end of chapter 1. Referrals now depend on the waiting time, which depends on the list itself (the grey arrow from the stock), closing a balancing loop.

## Learning outcomes

By the end of this chapter you will be able to:

1. **Extract** variables, with units and sources, from interview notes (Try it 2).
2. **Write** causal links with a polarity and a source as an edge list, and **check** it with code (Try it 3).
3. **Find** every feedback loop in an edge list and **classify** each as reinforcing or balancing (Try it 4).
4. **Draw** a causal loop diagram from the edge list (Try it 5).
5. **Describe** a reference mode from data and **justify** a time horizon (Try it 6).
6. **Build** a model boundary chart and **defend** what it leaves out (Try it 7).
7. **State** a dynamic hypothesis and **test** its predictions against data (Try it 8).

## 1. Start from the problem, not the system

A model is built to answer a question, and the question decides what goes in. "Model the hospital" has no end. "Why has the outpatient list doubled in ten years despite repeated extra activity, and what would bring the wait down by 2028?" does: it names a quantity, a pattern over time, a horizon and a decision. Write the question down first, in one or two sentences, and agree it with whoever will use the answer. Every later choice (variables, boundary, time step) is tested against it [@sterman2000; @martinezmoyano2013].

Our client is the hospital group's chief operating officer. The question, in her words: *the list keeps growing whatever we do; what should we do differently?*

## 2. Listening: from interview notes to variables

The modeller's first evidence is what people who work in the system say about it. Below are notes from four short interviews. They are invented for this book, but they are typical of what such interviews produce: specific, partial, sometimes contradictory, and full of cause and effect. Storing them as data, a list of dictionaries, means every variable and link can point back to who said it.

In [ ]:
notes = [
    {"who": "Consultant", "said":
     "We run every clinic at full stretch. When the list goes up, management "
     "asks for extra Saturday clinics and we do them, but you can only do "
     "that for so long. Two of our registrars left last year, burnt out, "
     "and their posts took nine months to fill. And patients arrive further "
     "along than they used to: a knee we could have managed with physio "
     "two years ago now needs surgery and three follow-up visits."},
    {"who": "Manager", "said":
     "The board watches one number, the patients waiting over a year. When "
     "it climbs, we buy appointments from private providers, and it comes "
     "down. But that money comes out of the same budget, so next year we "
     "have less to hire our own staff. A validation exercise takes a few "
     "thousand off the list, and then it creeps back."},
    {"who": "GP", "said":
     "When the orthopaedic wait was six months, I referred anyone who might "
     "benefit. Now I tell people it is two years, and many say there is no "
     "point, or pay to go private if they can. The referrals you see are "
     "not the need that is out there."},
    {"who": "Patient advocate", "said":
     "Being on the list is not neutral. People get worse while they wait, "
     "they lose work, they come to us in pain. Some are taken off the list "
     "by a letter they never saw, and the hospital counts it as progress. "
     "And the population gets older every year, so need only goes one way."},
]
for n in notes:
    print(f"{n['who']} ({len(n['said'].split())} words)")

Reading the notes, underline every quantity that could go up or down. Then give each one a name that works as a variable:

- **A noun, not a verb or an event.** "Staff", not "hiring problems"; "Overtime", not "doing Saturday clinics".
- **Neutral in direction.** "Staff", not "staff shortage", so that "more" has one meaning and the polarity of each link is clear.
- **Measurable in principle, with units.** If you cannot say what units it has, you do not yet know what it is.
- **Classed as a stock, a flow or an auxiliary.** Stocks are what you could count if time stopped (patients waiting, staff, fatigue); flows change them; auxiliaries are worked out from the others.

The table below is the result for these notes. `source` records who mentioned each variable, and `measured` whether the data in this book contains it.

In [ ]:
variables = pd.DataFrame([
    # name, kind, units, source, measured
    ("Need in the population", "auxiliary", "patients/month",
     "Patient advocate", "no"),
    ("Referrals", "flow", "patients/month", "GP", "no"),
    ("Waiting list", "stock", "patients", "all four", "yes"),
    ("Waiting time", "auxiliary", "months", "GP", "partly"),
    ("Leaving the list", "flow", "patients/month",
     "GP, Manager, Patient advocate", "no"),
    ("Severity", "stock", "index (2014 = 1)",
     "Consultant, Patient advocate", "no"),
    ("Appointments per patient", "auxiliary", "appointments/patient",
     "Consultant", "no"),
    ("Treatments", "flow", "patients/month", "Consultant", "no"),
    ("Capacity", "auxiliary", "appointments/month", "Consultant", "no"),
    ("Pressure", "auxiliary", "dimensionless", "Manager, Consultant", "no"),
    ("Overtime", "auxiliary", "extra clinics/month", "Consultant", "no"),
    ("Fatigue", "stock", "index (rested = 0)", "Consultant", "no"),
    ("Staff leaving", "flow", "clinicians/month", "Consultant", "no"),
    ("Staff", "stock", "clinicians", "Consultant, Manager", "no"),
    ("Outsourcing", "flow", "patients/month", "Manager", "no"),
    ("Budget for own staff", "auxiliary", "euros/year", "Manager", "no"),
], columns=["variable", "kind", "units", "source", "measured"])
print(variables.to_string(index=False))
print()
print(variables["kind"].value_counts().to_string())

Sixteen variables from four short interviews, and only one of them, the waiting list, is in the data (waiting time is partly there, through the time bands). That is normal. A model's structure comes mostly from what people know about how the system works, and only partly from numbers [@forrester1961]. Notice also the units: capacity is in appointments a month, treatments in patients a month, and appointments per patient converts one into the other, the Conversion molecule of chapter 1.

**Try it 2.**

- *Check.* Which four variables are stocks, and why is `Severity` one of them?
- *Apply.* The advocate says patients "lose work". Name a variable for this, give its kind and units, and decide whether it belongs in a model whose question is about the waiting time.
- *Extend.* The manager talks about "the patients waiting over a year". Is that a new variable, or a part of one already in the table? What would you need to add to the model to represent it?

<details><summary>Worked answer</summary>

*Check.* `Waiting list`, `Severity`, `Fatigue` and `Staff` (the cell below lists them). Severity is a stock because it is a state of the people waiting that builds up over time and persists: the average condition of patients on the list changes only as people deteriorate, join or leave.

*Apply.* "Patients out of work", a stock in patients (or an auxiliary, a fraction of the list). It matters to patients and to the economy, but it does not feed back on referrals, capacity or treatment in our question, so it is an output to report, not a cause to model. A variable can be important and still be outside the loops.

*Extend.* It is part of the waiting list: the stock split by time waited, which the data already has in four bands. Representing it needs the list as a chain of stocks that patients age through, the Aging Chain of chapter 4, Delays, pipelines and memory.
</details>

In [ ]:
print("Stocks:", ", ".join(variables.loc[variables["kind"] == "stock",
                                         "variable"]))

## 3. Links and polarity

A **causal link** says that one variable affects another: Waiting time affects Referrals. Its **polarity** says how. A link from X to Y is **positive** (+) if, all else equal, an increase in X makes Y higher than it would otherwise have been, and a decrease makes it lower. It is **negative** (−) if an increase in X makes Y lower than it would otherwise have been. In symbols, for an auxiliary $Y = f(X, \dots)$ the polarity is the sign of $\partial Y / \partial X$. For a stock it is the sign with which the flow enters: an inflow is a + link to its stock, an outflow a − link, because an outflow that rises makes the stock lower than it would otherwise have been, even if the stock is still growing.

"Than it would otherwise have been" matters. Treatments are a − link to the waiting list, yet the list can rise while treatments rise; the link says only that the list is lower than it would be with fewer treatments [@sterman2000].

We write the links as an **edge list**, a table with one row per link: cause, effect, polarity, whether the link involves a significant **delay**, and who told us about it. An edge list is easy to read, to check with code, to keep under version control and to turn into a picture.

In [ ]:
edges = pd.DataFrame([
    # cause, effect, sign, delay, source
    ("Need in the population", "Referrals", "+", False, "Patient advocate"),
    ("Referrals", "Waiting list", "+", False, "inflow"),
    ("Treatments", "Waiting list", "-", False, "outflow"),
    ("Waiting list", "Waiting time", "+", False, "definition"),
    ("Waiting time", "Referrals", "-", False, "GP"),
    ("Waiting time", "Leaving the list", "+", False, "GP"),
    ("Leaving the list", "Waiting list", "-", False, "outflow"),
    ("Waiting time", "Severity", "+", True, "Patient advocate"),
    ("Severity", "Appointments per patient", "+", False, "Consultant"),
    ("Appointments per patient", "Treatments", "-", False, "Consultant"),
    ("Capacity", "Treatments", "+", False, "Consultant"),
    ("Waiting list", "Pressure", "+", False, "Manager"),
    ("Pressure", "Overtime", "+", False, "Consultant"),
    ("Overtime", "Capacity", "+", False, "Consultant"),
    ("Overtime", "Fatigue", "+", True, "Consultant"),
    ("Fatigue", "Staff leaving", "+", False, "Consultant"),
    ("Staff leaving", "Staff", "-", False, "outflow"),
    ("Staff", "Capacity", "+", False, "Consultant"),
    ("Pressure", "Outsourcing", "+", False, "Manager"),
    ("Outsourcing", "Treatments", "+", False, "Manager"),
    ("Outsourcing", "Budget for own staff", "-", True, "Manager"),
    ("Budget for own staff", "Staff", "+", True, "Manager"),
], columns=["cause", "effect", "sign", "delay", "source"])


def check_edges(edges, variables):
    """Raise an error if the edge list has a mistake a reader could miss."""
    known = set(variables["variable"])
    for col in ("cause", "effect"):
        unknown = set(edges[col]) - known
        assert not unknown, f"not in the variable table: {unknown}"
    assert edges["sign"].isin(["+", "-"]).all(), "every sign is + or -"
    assert not edges.duplicated(["cause", "effect"]).any(), "a link twice"
    assert (edges["cause"] != edges["effect"]).all(), "a link to itself"
    unused = known - set(edges["cause"]) - set(edges["effect"])
    assert not unused, f"variables with no links: {unused}"
    return f"{len(edges)} links between {len(known)} variables: all checked"


print(check_edges(edges, variables))
print(edges["sign"].value_counts().to_string())
print(f"links with a delay: {edges['delay'].sum()}")

Twenty-two links, every one of them traceable to an interview, a definition or the stock-and-flow accounting. Six are negative. Four carry a delay: patients take time to deteriorate, fatigue builds slowly, a budget cut bites next year, and a vacant post takes months to fill. The check passes; it would fail if a name were misspelt, a sign missing or a variable left floating, the mistakes that are easy to make in a drawing tool and hard to see.

**Try it 3.**

- *Check.* Why is the link from `Treatments` to `Waiting list` negative even in a year when both rise?
- *Apply.* The manager says validation "takes a few thousand off the list". Validation is done when pressure is high. Add the link from `Pressure` to `Leaving the list` with its polarity and source, and rerun `check_edges`.
- *Extend.* Waiting time is roughly the list divided by treatments. What is the polarity of a link from `Treatments` to `Waiting time`? We left it out; what would including it change?

<details><summary>Worked answer</summary>

*Check.* Polarity compares with what would otherwise have happened. More treatments always leave the list lower than it would have been with fewer, whatever else is pushing it up. In a year when both rise, referrals rose faster.

*Apply.* `("Pressure", "Leaving the list", "+", False, "Manager")`: more pressure, more validation, more patients removed. The check passes with 23 links (the cell below).

*Extend.* Negative: more treatments, shorter wait. Including it is correct but adds many loops, because every path that reaches `Treatments` now also closes through `Waiting time` (Try it 4 counts them). We left it out to keep the diagram readable, since `Treatments` already reaches `Waiting time` through the list. Such choices should be recorded, as here, so a reviewer can disagree with them.
</details>

In [ ]:
validation = pd.DataFrame(
    [("Pressure", "Leaving the list", "+", False, "Manager")],
    columns=edges.columns)
edges_plus = pd.concat([edges, validation], ignore_index=True)
print(check_edges(edges_plus, variables))

## 4. Finding the loops with code

A **feedback loop** is a path of links that starts at a variable and comes back to it without visiting any variable twice. Its polarity follows from its links: count the negative links round the loop. An even number (including zero) makes a **reinforcing** loop (R), which amplifies a change; an odd number makes a **balancing** loop (B), which opposes it. Two negatives cancel, just as two minus signs multiply to a plus.

Finding loops by eye works for three or four. It fails quickly: the number of loops can grow much faster than the number of links, and people reliably miss the long ones, which are often the ones that matter. The function below finds every loop with a **depth-first search**: from each variable it follows links forward, keeping track of the path so far, and records a loop whenever a path returns to its start. To list each loop once, a search starting at a variable only visits variables later in alphabetical order.

In [ ]:
def find_loops(edges):
    """Every simple feedback loop in an edge list, with its polarity.

    Returns a list of (variables in order, polarity), where polarity is
    "R" (reinforcing: an even number of negative links) or "B" (balancing).
    """
    out = {}
    for cause, effect, sign in edges[["cause", "effect", "sign"]].itertuples(
            index=False):
        out.setdefault(cause, []).append((effect, sign))
    names = sorted(set(edges["cause"]) | set(edges["effect"]))
    loops = []

    def follow(start, here, path, signs):
        for nxt, sign in out.get(here, []):
            if nxt == start:                       # back where we began
                negatives = (signs + [sign]).count("-")
                loops.append((path, "R" if negatives % 2 == 0 else "B"))
            elif nxt > start and nxt not in path:  # each loop found once
                follow(start, nxt, path + [nxt], signs + [sign])

    for start in names:
        follow(start, start, [start], [])
    # start every loop at the waiting list, where it has one, for reading
    turned = []
    for path, polarity in loops:
        k = path.index("Waiting list") if "Waiting list" in path else 0
        turned.append((path[k:] + path[:k], polarity))
    return turned


loops = find_loops(edges)
print(f"{len(loops)} loops found")
for path, polarity in loops:
    print(polarity, " -> ".join(path + [path[0]]))

The search finds seven loops. Giving each a short name that says what it does makes the diagram something people can talk about. The names below are keyed by the set of variables in the loop, so if you change the edge list, any new loop shows up as unnamed and asks for a name.

In [ ]:
LOOP_NAMES = {
    frozenset({"Waiting list", "Waiting time", "Referrals"}):
        ("B1", "Discouraged referrals"),
    frozenset({"Waiting list", "Waiting time", "Leaving the list"}):
        ("B2", "Giving up"),
    frozenset({"Waiting list", "Pressure", "Overtime", "Capacity",
               "Treatments"}):
        ("B3", "Working harder"),
    frozenset({"Waiting list", "Pressure", "Outsourcing", "Treatments"}):
        ("B4", "Buying activity"),
    frozenset({"Waiting list", "Waiting time", "Severity",
               "Appointments per patient", "Treatments"}):
        ("R1", "Sicker by the time they are seen"),
    frozenset({"Waiting list", "Pressure", "Overtime", "Fatigue",
               "Staff leaving", "Staff", "Capacity", "Treatments"}):
        ("R2", "Burnout"),
    frozenset({"Waiting list", "Pressure", "Outsourcing",
               "Budget for own staff", "Staff", "Capacity", "Treatments"}):
        ("R3", "Borrowing from next year"),
}


def name_loops(loops):
    rows = []
    for path, polarity in loops:
        loop_id, name = LOOP_NAMES.get(frozenset(path), ("?", "unnamed"))
        assert loop_id == "?" or loop_id[0] == polarity, f"{name}: polarity"
        rows.append((loop_id, name, polarity, len(path),
                     " -> ".join(path)))
    table = pd.DataFrame(rows, columns=["id", "name", "polarity", "links",
                                        "path"])
    return table.sort_values("id").reset_index(drop=True)


loop_table = name_loops(loops)
print(loop_table[["id", "name", "polarity", "links"]].to_string(index=False))

Four balancing loops and three reinforcing ones. Read them as the story the interviews told:

- **B1 Discouraged referrals** and **B2 Giving up**: long waits deter referrals and push patients off the list. They hold the list in check, but by hiding need, not meeting it.
- **B3 Working harder** and **B4 Buying activity**: managers respond to a long list with overtime and outsourcing. These are the intended corrections.
- **R1 Sicker by the time they are seen**: long waits make patients need more appointments each, which cuts the patients treated and lengthens the wait further.
- **R2 Burnout** and **R3 Borrowing from next year**: the same corrections, with a delay, erode the staff that capacity depends on. Each starts as part of a balancing loop and turns into a reinforcing one.

Note that `assert` in `name_loops` checks each name against the polarity the code computed, so a mislabelled loop cannot slip through.

**Try it 4.**

- *Check.* Every loop passes through `Waiting list`. Is that a coincidence?
- *Apply.* Add the validation link from Try it 3 and find the loops again. What new loop appears, and what is its polarity?
- *Extend.* Add the link `Treatments` to `Waiting time` (negative) to the original edges. How many loops are there now, and why so many more?

<details><summary>Worked answer</summary>

*Check.* No. The interviews were about the list, and every mechanism people described responds to it. But it also means the diagram has no loop that runs without the list, for example staff leaving because colleagues have left. Ask whether such loops exist before concluding they do not.

*Apply.* One new loop: Waiting list -> Pressure -> Leaving the list -> Waiting list, with one negative link, so balancing. Call it B5, Validation: it takes patients off the list without treating them, like B2.

*Extend.* Sixteen loops instead of seven (the cell below). Every loop that reached `Treatments` could already close through the list; now each can also close through `Waiting time`, and each of those can continue through B1, B2 or R1. One plausible link doubled the count. This is why loops should be found by code, and why diagrams for communication usually show only the main loops.
</details>

In [ ]:
print(name_loops(find_loops(edges_plus)).query("id == '?'")[
    ["polarity", "path"]].to_string(index=False))
waiting_time_link = pd.DataFrame(
    [("Treatments", "Waiting time", "-", False, "definition")],
    columns=edges.columns)
more = find_loops(pd.concat([edges, waiting_time_link], ignore_index=True))
print(f"with Treatments -> Waiting time: {len(more)} loops, "
      f"{sum(p == 'R' for _, p in more)} reinforcing")

## 5. Drawing the diagram

A diagram is how the loops reach other people. Drawing it from the edge list, rather than by hand, keeps the two in step: change a link and the picture changes with it. The function below places each variable at a position we choose, draws every link as a curved arrow with its polarity near the head, marks delays with a double bar across the arrow, and labels the loops. Negative links are dashed so the diagram also reads without colour. Positions are the one thing chosen by hand, because a good layout depends on the story you want to tell; a layout that separates the patient side (above the list) from the management side (below it) works well here.

In [ ]:
POS = {
    "Need in the population": (-8.0, 0.0), "Referrals": (-4.6, 0.0),
    "Waiting list": (0.0, 0.0), "Waiting time": (-2.4, 2.6),
    "Leaving the list": (1.9, 2.6), "Severity": (-2.4, 5.2),
    "Appointments per patient": (4.2, 5.2), "Treatments": (4.2, 0.0),
    "Pressure": (0.0, -2.6), "Outsourcing": (4.2, -2.6),
    "Overtime": (-2.0, -5.0), "Capacity": (8.4, -1.0),
    "Fatigue": (0.6, -4.6), "Staff leaving": (3.2, -5.2),
    "Budget for own staff": (7.8, -3.4), "Staff": (7.2, -5.8),
}
BEND = {("Overtime", "Capacity"): 0.55, ("Waiting time", "Referrals"): 0.0,
        ("Leaving the list", "Waiting list"): 0.0}
LOOP_LABELS = {"B1": (-2.4, 0.9), "B2": (0.0, 1.7), "R1": (0.9, 4.2),
               "B3": (-0.3, -4.0), "B4": (2.1, -1.3), "R2": (3.2, -3.9),
               "R3": (5.8, -1.7)}


def curve(p0, p2, bend, s):
    """Point at fraction s along the arc matplotlib draws for arc3,rad=bend."""
    p0, p2 = np.asarray(p0, float), np.asarray(p2, float)
    dx, dy = p2 - p0
    c = (p0 + p2) / 2 + bend * np.array([dy, -dx])
    return (1 - s) ** 2 * p0 + 2 * (1 - s) * s * c + s ** 2 * p2


def draw_cld(ax, edges, pos, loop_labels=None, bend=None, default_bend=0.12):
    """Draw a causal loop diagram from an edge list and node positions."""
    bend = bend or {}
    ax.set_aspect("equal")
    ax.axis("off")
    boxes = {}
    for name, (x, y) in pos.items():
        text = name.replace(" per ", "\nper ")
        boxes[name] = ax.text(x, y, text, ha="center", va="center",
                              fontsize=9, zorder=2,
                              bbox=dict(boxstyle="round,pad=0.25", fc="white",
                                        ec="none"))
    for cause, effect, sign, delay in edges[["cause", "effect", "sign",
                                             "delay"]].itertuples(index=False):
        r = bend.get((cause, effect), default_bend)
        ax.annotate("", xy=pos[effect], xytext=pos[cause], zorder=3,
                    arrowprops=dict(
                        arrowstyle="-|>", lw=1, color="black",
                        ls="-" if sign == "+" else (0, (4, 2)),
                        connectionstyle=f"arc3,rad={r}",
                        patchA=boxes[cause].get_bbox_patch(),
                        patchB=boxes[effect].get_bbox_patch(),
                        shrinkA=2, shrinkB=2))
        # polarity: beside the arc, near the arrow head
        here = curve(pos[cause], pos[effect], r, 0.72)
        along = curve(pos[cause], pos[effect], r, 0.74) - curve(
            pos[cause], pos[effect], r, 0.70)
        side = np.array([-along[1], along[0]]) / np.hypot(*along)
        ax.text(*(here + 0.3 * side), "+" if sign == "+" else "\u2212",
                ha="center", va="center", fontsize=11, fontweight="bold",
                zorder=4)
        if delay:                       # two short bars across the arc
            mid = curve(pos[cause], pos[effect], r, 0.45)
            for k in (-0.09, 0.09):
                c = mid + k * along / np.hypot(*along)
                a, b = c - 0.28 * side, c + 0.28 * side
                ax.plot([a[0], b[0]], [a[1], b[1]], color="black", lw=1.6,
                        ls="-", zorder=4)
    for label, (x, y) in (loop_labels or {}).items():
        ax.text(x, y, label, ha="center", va="center", fontsize=10,
                fontweight="bold", zorder=4,
                bbox=dict(boxstyle="circle,pad=0.2", fc="white", ec="black"))
    xs, ys = zip(*pos.values())
    ax.set_xlim(min(xs) - 1.8, max(xs) + 1.8)
    ax.set_ylim(min(ys) - 2.2, max(ys) + 0.8)


fig, ax = plt.subplots(figsize=(11, 8))
draw_cld(ax, edges, POS, LOOP_LABELS, BEND)
ax.set_title("Why the waiting list keeps growing: a causal loop diagram")
plt.show()

Figure: causal loop diagram of the waiting list, drawn from the edge list. Solid arrows are positive links, dashed arrows negative ones; the sign beside each arrow head repeats the polarity. A double bar across an arrow marks a delay. Circled labels identify the loops: B1 Discouraged referrals, B2 Giving up, B3 Working harder, B4 Buying activity, R1 Sicker by the time they are seen, R2 Burnout, R3 Borrowing from next year.

A causal loop diagram is a map of hypotheses, not a model. It does not say how strong each link is, how long each delay lasts or which loop dominates. It cannot even say whether the list will rise or fall: B3 and R2 share their first links, and only numbers can say which wins. Those questions need the stocks, flows and equations of chapters 3 to 7. What the diagram does well is make the team's assumptions visible, so they can be argued with before anyone writes an equation.

**Try it 5.**

- *Check.* Which links are dashed, and how many are there?
- *Apply.* Draw a smaller diagram with only the patient side: the links of loops B1, B2 and R1. Filter `edges` to rows whose cause and effect both lie in those loops, and pass only their positions.
- *Extend.* Delays matter most in balancing loops. Which balancing loops contain a delay, and which reinforcing loops do? What does that suggest about how the management responses behave over time?

<details><summary>Worked answer</summary>

*Check.* Six: Treatments to Waiting list, Waiting time to Referrals, Leaving the list to Waiting list, Appointments per patient to Treatments, Staff leaving to Staff, and Outsourcing to Budget for own staff. The cell below lists them from the edge list, which is quicker and safer than counting dashes in a picture.

*Apply.* The cell below draws it. With fewer variables the layout can be tighter, and the three loops can be explained one at a time.

*Extend.* No balancing loop in this diagram contains a delay; R1, R2 and R3 each contain at least one. So the management responses (B3, B4) act quickly, while their side effects (R2, R3) and the deterioration of waiting patients (R1) arrive later. That is the signature of a policy that works at first and then disappoints, a pattern chapter 7, Patterns that repeat, calls a fix that fails.
</details>

In [ ]:
print(edges.loc[edges["sign"] == "-", ["cause", "effect"]].to_string(
    index=False))

patient_side = set().union(*[set(p) for p, _ in loops
                             if LOOP_NAMES[frozenset(p)][0]
                             in ("B1", "B2", "R1")])
small = edges[edges["cause"].isin(patient_side)
              & edges["effect"].isin(patient_side)]
fig, ax = plt.subplots(figsize=(8, 6))
draw_cld(ax, small, {k: POS[k] for k in patient_side},
         {k: LOOP_LABELS[k] for k in ("B1", "B2", "R1")}, BEND)
ax.set_title("The patient side of the diagram")
plt.show()

for path, polarity in loops:
    loop_id = LOOP_NAMES[frozenset(path)][0]
    pairs = set(zip(path, path[1:] + path[:1]))
    has_delay = edges[[(c, e) in pairs for c, e in
                       zip(edges["cause"], edges["effect"])]]["delay"].any()
    print(f"{loop_id}: delay {'yes' if has_delay else 'no'}")

Figure: the patient side of the causal loop diagram: loops B1 Discouraged referrals, B2 Giving up and R1 Sicker by the time they are seen.

## 6. The reference mode

A **reference mode** is the pattern over time that the model must explain: a graph of the problem variables, drawn from data where there is data and from people's descriptions where there is not, together with the futures people hope for and fear. It turns "the list is too long" into "the list grew steadily for six years, jumped in one, and has been flat for two", which a model can be tested against. The **time horizon** is how far back and forward the graph runs. It must reach back far enough to show how the problem arose, and forward far enough for the slowest delays in the diagram to play out.

In [ ]:
wl_m = wl.set_index("month")
total = wl_m["total"]
share = 100 * wl_m["w12_plus"] / total
months = np.arange(len(total))                       # 0 = January 2014

pre = total[:"2019-12"]
slope, intercept = np.polyfit(months[:len(pre)], pre.to_numpy(), 1)
rate = np.exp(12 * np.polyfit(months[:len(pre)], np.log(pre), 1)[0]) - 1
future = pd.date_range("2024-01-01", "2028-12-01", freq="MS")
ahead = np.arange(len(total), len(total) + len(future))
latest = total.iloc[-1]
feared = latest + slope * (ahead - ahead[0] + 1)     # pre-2020 trend resumes
hoped = latest + (pre.iloc[-1] - latest) * (ahead - ahead[0] + 1) / len(ahead)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
a1.plot(total.index, total / 1000, label="Data")
a1.plot(total.index[:len(pre)],
        (intercept + slope * months[:len(pre)]) / 1000,
        label="Trend, 2014 to 2019")
a1.plot(future, feared / 1000, label="Feared: the old trend resumes")
a1.plot(future, hoped / 1000, label="Hoped: back to the 2019 level by 2028")
a1.axvspan(future[0], future[-1], color="grey", alpha=0.12)
a1.set_title("Reference mode: patients waiting")
a1.set_ylabel("Patients waiting (thousands)")
a1.legend(loc="upper left", fontsize=8)
a2.plot(share.index, share, color="#D55E00", ls="--")
a2.axvspan(future[0], future[-1], color="grey", alpha=0.12)
a2.set_title("Share waiting more than a year")
a2.set_ylabel("Share of the list (%)")
a2.set_xlabel("Month")
plt.show()

print(f"Trend 2014 to 2019: {slope:,.0f} patients a month "
      f"({rate:.1%} a year)")
print(f"Step: {total['2019-12'].iloc[0]:,} in December 2019, "
      f"{total['2021-12'].iloc[0]:,} in December 2021")
flat = np.polyfit(np.arange(24), total["2022":].to_numpy(), 1)[0]
print(f"Since January 2022: {flat:,.0f} patients a month")
print(f"Over a year: {share.iloc[0]:.0f}% in January 2014, "
      f"{share['2019-12'].iloc[0]:.0f}% in December 2019, "
      f"{share.iloc[-1]:.0f}% in December 2023")

Figure: the reference mode. Top: patients waiting, with a straight-line trend fitted to 2014 to 2019 and two futures for 2024 to 2028 (shaded): the trend resuming, which managers fear, and a return to the December 2019 level, which they hope for. Bottom: the share of patients waiting more than a year.

In words, the reference mode has three phases. From 2014 to 2019 the list grew steadily, by about 158 patients a month (5.2% a year). In 2020 and 2021 it jumped from 43,839 to 68,552 as activity collapsed and recovered. Since January 2022, with extra activity funded, it has been flat, rising by about 43 a month. Throughout, the share waiting more than a year rose: from 29% to 39% before the shock and to 50% after it. The list is not only longer; it is older.

The time horizon is January 2014 to December 2028, in months. Ten years back shows the steady growth before the shock, so the model must explain more than the shock. Five years ahead is long enough for the slowest delays in the diagram, training and hiring staff and the next year's budget, to work through, and matches the question's 2028 target.

**Try it 6.**

- *Check.* How much did the list grow in percent per year from 2014 to 2019?
- *Apply.* Had the pre-2020 trend simply continued, with no shock, in which year would the list have reached 70,000?
- *Extend.* In which month did the share waiting more than a year first reach 40%? What does the timing tell you about how the shock fed into the long waits?

<details><summary>Worked answer</summary>

*Check.* 5.2% a year, from a log-linear fit to the 72 months (the cell above prints it).

*Apply.* The trend starts at about 31,100 in January 2014 and adds 158 a month, so it reaches 70,000 after about 246 months, in 2034. The shock brought the list to that level more than ten years early.

*Extend.* August 2020, five months after the shock began (the cell below). Patients who joined in March 2020 could not yet have waited a year by then: the share rose first because fewer people were seen, so everyone already waiting aged together, and kept rising into 2022 as the shock's patients crossed a year. A change in a stock shows up in its age profile with a delay, which is why chapter 4 models the list as a chain of time bands.
</details>

In [ ]:
years_to_70k = (70000 - intercept) / slope / 12
print(f"trend reaches 70,000 in {2014 + years_to_70k:.1f}")
print("share first at 40% or more:",
      share[share >= 40].index[0].strftime("%B %Y"))

## 7. The model boundary

Every model leaves almost everything out. A **model boundary chart** says what is in and what is not, in three columns [@sterman2000]:

- **Endogenous**: variables the model works out, inside its feedback loops.
- **Exogenous**: variables that drive the model from outside; the model uses them but nothing in the model changes them.
- **Excluded**: things deliberately left out, each with a reason.

The test of a boundary is whether something treated as exogenous really responds to what happens inside. If it does, leaving it out can make a policy look good in the model and fail in the world. Part of the chart can be read off the edge list: variables in at least one loop are endogenous, and variables with no incoming links are exogenous. The rest is judgement, and goes in with its reasons.

In [ ]:
in_loops = set().union(*[set(p) for p, _ in loops])
no_causes = set(edges["cause"]) - set(edges["effect"])
endogenous = sorted(in_loops)
exogenous = sorted(no_causes) + [
    "Government budget for the group",
    "Shock to activity in 2020",
    "Funding for extra activity from 2022",
    "Minimum treatment time",
]
excluded = {
    "Individual specialties": "the question is about the whole list",
    "Private sector capacity and prices": "outsourcing is small next to "
                                          "private demand",
    "Emergency and inpatient care": "separate budgets and lists",
    "GP workload": "affects referrals only through waits, already in B1",
    "Clinical priority within the list": "added in chapter 4 with the "
                                         "time bands",
}
boundary = pd.DataFrame({
    "Endogenous": pd.Series(endogenous),
    "Exogenous": pd.Series(exogenous),
    "Excluded": pd.Series([f"{k} ({v})" for k, v in excluded.items()]),
}).fillna("")
print(f"{len(endogenous)} endogenous, {len(exogenous)} exogenous, "
      f"{len(excluded)} excluded")
boundary

Fifteen of the sixteen variables in the diagram are endogenous: every one except `Need in the population` sits in at least one loop. Four more exogenous drivers come from the data and the case history: the group's budget, the shock of 2020, the funding from 2022 and the minimum treatment time. Five things are excluded, each with a reason that someone could challenge. The chart is short enough to put in front of a client, which is its purpose: it says, before any results, what the model can and cannot speak to.

**Try it 7.**

- *Check.* Which variable in the diagram is exogenous, and how does the code know?
- *Apply.* The advocate says the population gets older every year. Is ageing endogenous or exogenous for this question? Could the waiting list ever affect it?
- *Extend.* "Private sector capacity" is excluded. Describe a feedback loop that would force it inside the boundary, and say what evidence would tell you it had become important.

<details><summary>Worked answer</summary>

*Check.* `Need in the population`: it is a cause in the edge list but never an effect, so nothing in the model changes it (the cell below).

*Apply.* Exogenous. The list does not change how old the population is over five years. Long waits could affect health and so the need for care, but that is a different variable (need), and it would enter through a new link, not through ageing.

*Extend.* If outsourcing grew large, private providers would hire more clinicians, some of them from the group's own staff, and their prices would rise. Outsourcing -> private demand for staff -> Staff leaving (+) -> Staff -> Capacity -> Treatments -> Waiting list -> Pressure -> Outsourcing is a reinforcing loop. Evidence: a rising share of leavers going to private providers, or rising prices for outsourced appointments.
</details>

In [ ]:
print("Causes that are never effects:", sorted(no_causes))

## 8. The dynamic hypothesis

A **dynamic hypothesis** is a statement of which feedback structure produces the reference mode. It is called dynamic because it explains a pattern over time, and a hypothesis because it could be wrong. Ours, in four sentences:

> Underlying demand grows faster than the capacity the group can sustain, so the list grows. The intended corrections, working harder (B3) and buying activity (B4), act quickly but wear down staff and the staff budget (R2, R3) after a delay, so their effect fades. Long waits are held in check mainly by deterring referrals and pushing patients off the list (B1, B2), which hides need rather than meeting it. And waiting itself makes patients need more care (R1), so every rise in the wait makes the next one easier.

A good hypothesis makes predictions that the data could contradict. The list alone cannot test every part of ours, but it can test some. If demand outgrew capacity before 2020, the net flow (the monthly change in the list, chapter 1) should have been positive and rising. If B1 and B2 are strong, extra activity should stop the list growing without making it fall far. If R1 and the long-wait loops matter, the list should keep getting older even when its size is flat.

In [ ]:
net = total.diff()["2014-02":"2019-12"]
years = (np.arange(len(net)) + 1) / 12
trend, start = np.polyfit(years, net.to_numpy(), 1)
recent = total.diff()["2022":]
checks = pd.DataFrame([
    ("Net flow positive and rising before 2020",
     f"net flow {start + trend:,.0f}/month in 2014 to "
     f"{start + trend * 6:,.0f} in 2019",
     "consistent" if trend > 0 and start > 0 else "contradicted"),
    ("Extra activity stops growth but the list barely falls",
     f"{(recent < 0).sum()} of {len(recent)} months fell; "
     f"net change {recent.sum():+,.0f}",
     "consistent" if abs(recent.sum()) < 0.05 * total.iloc[-1]
     else "contradicted"),
    ("The list keeps ageing while its size is flat",
     f"over a year: {share['2021-12'].iloc[0]:.0f}% in Dec 2021, "
     f"{share.iloc[-1]:.0f}% in Dec 2023",
     "consistent" if share.iloc[-1] >= share["2021-12"].iloc[0]
     else "contradicted"),
], columns=["prediction", "what the data shows", "verdict"])
print(checks.to_string(index=False))

All three predictions are consistent with the data: the net flow rose from about 86 to 278 patients a month between 2014 and 2019; the list fell in only 8 of the 24 months after the extra activity began and was 1,375 patients longer at the end; and the share waiting over a year held at 49% to 50% while the total barely moved. That is not proof. Each pattern has other possible explanations, and the data cannot say which loops are strong, because no flows were observed. What the checks do is rule out the simplest rival hypothesis, that the list is long only because of the 2020 shock: the list was growing, and its growth was speeding up, for six years before it.

**Try it 8.**

- *Check.* What is the slope of the net-flow trend, in patients a month per year, before 2020?
- *Apply.* What data would let you test B1, Discouraged referrals, directly? Say what you would look for.
- *Extend.* A colleague proposes a rival hypothesis: "capacity has been flat and referrals have grown at a steady rate since 2014; there are no feedback loops that matter." What would this predict for the list after 2022, when activity rose by about a tenth? Does the data favour it or ours?

<details><summary>Worked answer</summary>

*Check.* About 38 patients a month per year: the gap between inflow and outflow widened by that much each year (the cell below).

*Apply.* Monthly referral counts by specialty and GP practice, alongside the published wait for each specialty. B1 predicts that referrals fall, relative to the population, where and when waits are long, and recover after waits fall. A rise in referrals after the 2022 initiative would be direct evidence.

*Extend.* With no feedback, a step of about 10% in capacity would turn the net flow strongly negative, and the list would fall by hundreds of patients a month, for years, as in the open-loop run of chapter 1. The data shows the list flat. That favours a hypothesis with balancing loops on the demand side (or one in which demand also jumped in 2022). Chapter 10, Data, calibration and uncertainty, makes this comparison formally by fitting both models.
</details>

In [ ]:
print(f"net-flow trend: {trend:.1f} patients a month, per year")

## Check it

Four habits keep a qualitative model honest. The cell below runs them on our edge list.

1. **Every link has a polarity and a source.** If nobody told you about a link and it is not a definition or an accounting identity, ask why it is there.
2. **Every variable has units.** A variable without units is not yet defined.
3. **Every loop contains a stock.** A loop made only of auxiliaries would be a set of simultaneous equations with no time in it; real feedback takes time because it passes through something that accumulates.
4. **Every loop has a name, and its name matches its polarity.**

In [ ]:
stocks = set(variables.loc[variables["kind"] == "stock", "variable"])
assert edges["source"].str.len().gt(0).all(), "a link with no source"
assert variables["units"].str.len().gt(0).all(), "a variable with no units"
for path, polarity in loops:
    assert stocks & set(path), f"no stock in {path}"
named = name_loops(loops)
assert (named["id"] != "?").all(), "an unnamed loop"
print(f"{len(edges)} links with sources, {len(variables)} variables with "
      f"units, {len(loops)} loops each with a stock and a name: all pass")

## Running case

This chapter's increment turns the case from one loop into a qualitative model with a stated purpose. The question is why the list keeps growing despite extra activity, and what would bring the wait down by 2028. The evidence is four interviews and the stock series. The products are an edge list of 22 links between 16 variables, seven named loops (B1 to B4, R1 to R3), a reference mode with three phases (steady growth, a shock, a plateau with an ageing list) over a horizon of January 2014 to December 2028, a boundary chart, and a dynamic hypothesis whose testable predictions the data does not contradict.

Chapters 3 to 7 turn the loops into stock-and-flow structure one at a time: growth in need and goal seeking in chapter 3, the list as an ageing chain and perceived waits in chapter 4, the treatment and staffing decisions in chapter 5, hiring and its swings in chapter 6, and outsourcing and overtime, R2 and R3, in chapter 7.

## Tell the decision maker

The diagram says three things a manager can act on. First, the list is held where it is by waits deterring referrals and patients giving up; a shorter wait will bring some of that hidden demand back, so extra capacity will shorten waits more than it shortens the list. Second, overtime and outsourcing work at first but, if sustained, wear down the staff and the staff budget they depend on; their benefit should be judged over two or three years, not two or three months. Third, every month of waiting makes patients more costly to treat; long waits are not only a symptom but also a cause. These are hypotheses, not findings, and the next chapters build the model to test how strong each one is.

## Going further

- **Group model building.** In practice the diagram is often built with the stakeholders in a facilitated workshop, not by the analyst from notes. The edge-list approach here works well as the record of such a workshop.
- **Larger graphs.** For diagrams with hundreds of links, a graph library such as networkx can find cycles, layouts and loop statistics; the depth-first search above is the same idea, written out.
- **Loop dominance.** Which loop drives the behaviour at each moment cannot be read from a diagram. Methods for measuring it from simulation runs come in chapters 3 and 8.
- **Drafting diagrams with language models.** Chapter 12, Models that learn, asks a language model to draft an edge list from interview notes and compares it with one built by hand.

## Mini-project

Choose another service with a queue that has grown despite effort (a court backlog, a housing list, planning applications, a repair service).

1. Write four short invented stakeholder notes, from people who see the queue from different sides.
2. Build the variable table and the edge list, with sources, and run `check_edges`.
3. Find the loops with `find_loops`, name them, and draw the diagram with `draw_cld`.
4. Find or sketch a reference mode and choose a time horizon, with reasons.
5. Write a boundary chart and a dynamic hypothesis in no more than five sentences, with at least two predictions that data could contradict.

## Further reading

- @sterman2000, chapters 3 and 5, covers the modelling process, causal loop diagrams and their pitfalls in depth.
- @senge1990 popularised loop thinking for managers and is a good source of loop stories.
- @meadows2008 explains feedback and leverage points without equations.
- @martinezmoyano2013 sets out agreed good practice for each step, including problem definition and boundaries.
- @homer2006 shows how the same qualitative steps are used in public health projects.